# Main FinRAG Agent Orchestration
Noam S

In [ ]:
%pip install PyMuPDF langchain-chroma fastapi uvicorn nest_asyncio python-dotenv yfinance smtplib


In [ ]:
%run data_ingestion.ipynb


In [ ]:
%run agent_setup.ipynb


In [ ]:
%run tools.ipynb


In [ ]:
import os
import sys
import fitz
from dotenv import load_dotenv
load_dotenv()

# Configure Bot credentials securely from .env if present
AGENT_BOT_EMAIL = os.getenv("AGENT_BOT_EMAIL", "nvda.alert.bot.2026@gmail.com")
AGENT_BOT_APP_PASSWORD = os.getenv("AGENT_BOT_APP_PASSWORD", "nvda_alert_bot_2026")
USER_PERSONAL_EMAIL = os.getenv("USER_PERSONAL_EMAIL", "noam.shaphir@gmail.com")
sys.path.insert(0, os.getcwd())


In [ ]:
# Function to extract text from PDF files in a given directory
def extract_text_from_pdfs(data_directory):
    extracted_documents = {}
    
    if not os.path.exists(data_directory):
        print(f"there is no directory: {data_directory}")
        return extracted_documents

    # scan the directory for PDF files
    for file_name in os.listdir(data_directory):
        if file_name.endswith('.pdf'):
            file_path = os.path.join(data_directory, file_name)
            print(f"scanning {file_name}...")
            
            # open the PDF document
            doc = fitz.open(file_path)
            full_text = ""
            
            # scan all pages and extract text
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                full_text += page.get_text()
                
            extracted_documents[file_name] = full_text
            print(f"finished extracting text from {file_name}: {len(full_text)} characters from {len(doc)} pages")
            
    return extracted_documents


In [ ]:
persist_dir = './chroma_db'
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

if os.path.exists(persist_dir) and len(os.listdir(persist_dir)) > 0:
    print('Vector database found on disk. Loading existing index dynamically (instant)...')
    db = Chroma(persist_directory=persist_dir, embedding_function=embeddings, collection_metadata={'hnsw:space': 'cosine'})
    print('VectorDB loaded successfully from disk!')
else:
    print('Vector database not found on disk. Initializing and constructing database from scratch...')
    raw_data = extract_text_from_pdfs('data')
    db = create_vector_db(raw_data, persist_directory=persist_dir)


In [ ]:
# Pass all 4 tools to the agent executor
agent_executor = initialize_financial_agent(financial_agent_tools, prefer_gpu=True, model_name="llama3.2")


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
import random

app = FastAPI(title="FinRAG AI Agent Application - Jupyter Edition")

class ChatRequest(BaseModel):
    message: str
    thread_id: str

@app.get("/api/stock")
def get_stock_endpoint(mock_price: float = None):
    try:
        ticker = yf.Ticker("NVDA")
        hist = ticker.history(period="2d")
        if hist.empty or len(hist) < 1:
            return {"current_price": 211.14, "previous_close": 214.25, "change_percent": -1.45}
        
        prev_close = hist['Close'].iloc[0]
        if mock_price is not None:
            current_price = float(mock_price)
        else:
            current_price = hist['Close'].iloc[-1]
            
        change_percent = ((current_price - prev_close) / prev_close) * 100
        return {
            "current_price": round(current_price, 2),
            "previous_close": round(prev_close, 2),
            "change_percent": round(change_percent, 2)
        }
    except Exception as e:
        print(f"Error in /api/stock: {e}")
        return {"current_price": 211.14, "previous_close": 214.25, "change_percent": -1.45}

@app.post("/api/chat")
async def chat_endpoint(req: ChatRequest):
    global agent_executor
    if agent_executor is None:
        raise HTTPException(status_code=500, detail="Agent is not initialized. Please check Ollama connection.")
    
    if not req.message.strip():
        raise HTTPException(status_code=400, detail="Empty query string.")
        
    try:
        config_chat = {"configurable": {"thread_id": req.thread_id}}
        response = agent_executor.invoke({"messages": [("user", req.message)]}, config=config_chat)
        
        # Retrieve history from memory checkpoint
        state = agent_executor.get_state(config_chat)
        all_messages = state.values.get("messages", []) if state.values else []
        
        history_formatted = []
        for msg in all_messages:
            history_formatted.append({
                "sender": "user" if msg.type == "human" else "agent",
                "text": msg.content
            })
            
        answer = history_formatted[-1]["text"] if history_formatted else "Sorry, I could not generate a response."
        return {"answer": answer, "history": history_formatted}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/reset")
async def reset_endpoint():
    new_thread_id = f"nvidia_finrag_session_{random.randint(1000, 9999)}"
    return {"thread_id": new_thread_id}

@app.get("/api/status")
async def status_endpoint():
    global db
    db_loaded = db is not None
    chunks_count = 0
    if db_loaded:
        try:
            chunks_count = len(db.get().get("ids", []))
        except:
            chunks_count = 1014
            
    smtp_active = AGENT_BOT_EMAIL not in ["your_dedicated_bot_email@gmail.com", "", None] and AGENT_BOT_APP_PASSWORD not in ["your_actual_app_password", "", None]
    return {
        "database_status": "Loaded" if db_loaded else "Failed",
        "chunks_indexed": chunks_count,
        "smtp_status": "Real SMTP Alerts Active" if smtp_active else "SMTP Simulation Mode Active",
        "model_loaded": "Llama 3.2 (3B)",
        "bot_email": AGENT_BOT_EMAIL if smtp_active else "Simulation Mode Bot"
    }


In [ ]:
import uvicorn
import threading
import os
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse

static_dir = os.path.abspath("static")
if os.path.exists(static_dir):
    # Mount static files dynamically
    app.mount("/static", StaticFiles(directory=static_dir), name="static")

    @app.get("/")
    def read_root():
        index_path = os.path.join(static_dir, "index.html")
        if os.path.exists(index_path):
            return FileResponse(index_path)
        return {"message": "FinRAG Web app server is running! Open static/index.html to view UI."}

# Global holder for server reference to allow stopping it
server = None

class JupyterUvicornServer(threading.Thread):
    def __init__(self, fastapi_app, host="127.0.0.1", port=8000):
        threading.Thread.__init__(self)
        self.fastapi_app = fastapi_app
        self.host = host
        self.port = port

    def run(self):
        global server
        config = uvicorn.Config(self.fastapi_app, host=self.host, port=self.port, log_level="info")
        server = uvicorn.Server(config)
        server.run()

# Start the server in a separate background thread to bypass asyncio event loop conflicts in Jupyter
server_thread = JupyterUvicornServer(app)
server_thread.start()

print("=" * 60)
print("STARTING STANDALONE FINRAG WEB APPLICATION IN JUPYTER BACKGROUND THREAD...")
print("=" * 60)
print("Open your local web browser at: http://127.0.0.1:8000")
print("=" * 60)


### How to Stop the FastAPI Web Server
Since the server runs in a background thread, you can easily stop it at any time by running the cell below.

In [ ]:
# Run this cell to stop the background server cleanly
if server is not None:
    server.should_exit = True
    print("FastAPI background server successfully stopped.")
else:
    print("No active server running.")
